# IMX296 quad-fisheye calibration

End-to-end record of how the 4-camera rig was calibrated on 2026-08-28/29, including
what failed and why. Run top to bottom; every cell is the command that was actually used.

**The one result that shapes everything else:** the lenses are 1.78 mm, D190/H160 on 1/3",
i.e. a genuine **>180 degree** fisheye. Three independent confirmations - the vendor spec,
our own fits (~192 deg diagonal), and OmniNxt hard-coding `fov=190` for the same module class.

That rules out the equidistant model (`pinhole-equi` / Kannala-Brandt), which projects through
`x/z` and cannot express rays at or past 90 deg incidence. It also rules out feeding these
cameras to **cuVSLAM** directly, whose only fisheye model is that same equidistant one:
virtual-stereo rectification becomes required rather than preferred.

---

> ### ⚠ Hardware changed 2026-08-31 — this notebook is now the procedure, not the result
>
> The cameras were **reconnected**, the lenses **refocused**, and the trigger generator replaced
> (H7 → **F401**). Every measured number below therefore describes optics that no longer exist.
> They are kept as the **prior each new result is checked against** — see `§3R` of
> `openspec/changes/retarget-vo-to-imx296-rig/tasks.md`, and the full solver record archived at
> `config/calib/archive/20260828/`.
>
> Two consequences worth stating up front:
> - **No intrinsic may be carried across a port.** The module now on port c is probably not the
>   one that was there before, and intrinsics belong to the module, not the port.
> - The port **sequence** c/d/f/e *is* verified unchanged (2026-08-31, from the image overlaps),
>   so the ring order below still holds and no pair was swapped.

## 0. Hardware state the recordings depend on

Checked before every session. Three of these do **not** persist across a reboot or a power
cycle, by design - the cameras free-run by default and external trigger is opt-in:

| | why it matters |
|---|---|
| `trigger_mode=1` | without it the generator keeps pulsing, the sensors ignore it, **nothing logs an error**, and the only symptoms are AE gain-hunting and frame sets that are not sets |
| `jetson_clocks` | three concurrent streams collapse without it |
| trigger pulse width | in Fast Trigger mode the pulse width *is* the exposure. Argus reports 0.521 ms; the generator emits **4.986 ms**. Trusting Argus puts every stamp 2.2 ms out |
| the generator's own settings | **the F401 boots at its compiled-in defaults** (30.000 fps, 5000 us, `pol 1`) after any power cycle, and says nothing about it. Read them back, never assume |

**Trigger generator, since 2026-08-31: STM32F401** ("Black Pill"), replacing the H7 whose 3.3 V
rail failed short. It enumerates as **USB CDC on `/dev/ttyACM0`** where the H7 was reachable only
on the M110 UART `/dev/ttyTHS1`; `j106-trigctl.py` speaks to either unchanged. Verified
2026-08-31: same 30.000 Hz, same **1.0 us** worst inter-camera skew, same 0.01 us/s drift - all
four channels are compare outputs on one 32-bit counter, so the edges are one hardware event.

In [ ]:
# on the board (ssh tx2-eth)
sudo bash -c 'echo 1 > /sys/module/imx296/parameters/trigger_mode'
sudo jetson_clocks
python3 /home/nvidia/tools/j106-trigctl.py --port /dev/ttyACM0 status     # F401: USB CDC
#   clock=hse25-pll84   <- the crystal started; the HSI fallback is ~1% and lands on the frame rate
#   period_us=33333  polarity=active_low  ch1..4_exposure_us=5000  pulse_ns=4985740
#   -> exposure_us:=4986 for the capture node

# POLARITY IS NOT A LABEL YOU TRUST - measure it.  scripts/port/trigger_probe.py --sweep
# does exactly this and restores the exposure afterwards (leaving the rig at a probe's
# exposure silently biases every later stamp).  Argus must be free: stop csi_sender.sh
# and any ROS capture first, since Argus serves one consumer at a time. The F401 reports active_low where the H7
# reported active_high, and the two readings are compatible only if the pin inverts and the
# MEANING does not. Fix the scene and sweep the commanded exposure:
#   5000 -> 15000 -> 30000 us  gave mean raw 1001.2 -> 1108.1 -> 1270.7 (cam1, black ~736),
#   i.e. a straight line at 0.0107 / 0.0108 counts per us.
# Brightness RISES with commanded exposure, so the commanded value is the ASSERTED width.
# The complement hypothesis (exposure = period - pulse) predicts the opposite and is dead.

## 1. Recording: nine stages, on the board, in ROS

Four single-camera stages for intrinsics, four **adjacent pairs** for extrinsics, and - added
after 2026-08-29 - a ninth where **three or more cameras see the board at once**.

The pair order walks the rig, not the camera names: ports are c=front-left, d=front-right,
e=back-left, f=back-right, so neighbours are **c -> d -> f -> e**, i.e. cam1 -> cam2 -> cam4 -> cam3.
`cam2` and `cam3` are **diagonal** - solving that pair asks for overlap the rig does not have.

**Why stage 9 exists.** Four pairs solved from four separate recordings closed the ring to
3.63 deg / 9.2 mm, and a Monte-Carlo over each pair's own spread says random error would leave
only ~0.5 deg. So most of it is **systematic bias inside each recording** - invisible while every
constraint is pairwise, and absorbed silently by the ring closure. Three cameras on one board at
one instant is what makes it observable.

### Record with `record_calib_session.sh`, then scp and solve offline

**The recording must carry capture timestamps, and only the ROS path does.** A pair frame counts
only if both cameras saw the target *at the same instant*, and section 6 keys that on the frame's
**own header stamp** - matching on arrival order reported ZERO simultaneous pairs on a rig
triggered to 1 us. `argus_capture_node` stamps at the exposure midpoint (`SOF - exposure/2`,
docs/timestamps.md) and publishes `FrameMeta` alongside, so the correspondence is well-posed by
construction.

Anything that writes bare numbered images throws that away and forces a correspondence to be
*reconstructed* - by index, valid only while no camera has dropped a frame, and needing a
correction for the 2 s stagger between Argus session opens. Do not do it: the timestamps already
exist upstream.

**What this costs, and the mitigation.** rosbag2's single writer thread caps this path at ~9-11 Hz,
so images are decimated (`EVERY_N=8` -> 3.75 Hz, keyed on the trigger edge - section 2), and there
is **no live coverage grid** while you sweep. That is what left the first attempt 23% usable with
22 of 36 cells untouched at the periphery. So: **scp and run the coverage check BEFORE tearing the
setup down**, while the rig and board are still where they were, and re-record any stage that came
back short.

`scripts/stream/calib_sender.sh` + `calib_receiver.py` remain the alternative - full 30 Hz to host
NVMe with a live coverage grid, at the price of having no capture timestamps. Use them when the
sweep quality is the risk and the stage is single-camera (intrinsics need no correspondence at all).

### How to sweep — the currency is POSES, not frames

At 30 Hz with slow motion, consecutive frames are nearly identical: five seconds is ~150 frames
but perhaps 5-10 genuinely different views. **The frame rate does not shorten the sweep; it only
buys the freedom to discard**, which is what the selector does. What sets the duration is how long
it takes to physically move the board through the pose set.

Global shutter at 5 ms takes *blur* off the table - no rolling-shutter shear to bias corners, and
slow motion stays sharp. It does not reduce how many distinct poses are needed.

Three things set that pose set, and none of them get faster at 30 Hz:

| | why |
|---|---|
| **Tilt, 30-45 deg in BOTH axes** | the omni (Mei) fit has 9 free parameters - xi, fx, fy, cx, cy, 4 distortion - and focal length is strongly correlated with distortion. Tilt is what separates them. A sweep that stays fronto-parallel can cover the whole image and still leave those coupled: low reprojection error, wrong parameters |
| **The periphery** | a >180 deg lens is least constrained at the rim, and that is where 13-21 of 64 cells per camera came back short last time |
| **Two or three distances** | scale and the radial terms |

**Do not try to fill the frame with the board.** At 192 deg, filling the frame means the board is
very close, and near the rim it is seen at a grazing angle where detection degrades. Coverage is
the **union across frames**, not per-frame area - and Kalibr accepts an observation at **>= 7 tags**
(`minTagsForValidObs = max(rows,cols)+1`), which is exactly what lets a partly-visible board
constrain the extreme corners.

**Duration.** About **20-40 s of continuous slow sweeping per single-camera stage** - spiral
outward to the rim, rolling the board through tilt as you go - which at 3.75 Hz recorded gives
~75-150 frames and at 30 Hz gives ~600-1200 for the selector to thin to ~150-220. The **pair
stages take longer**: both cameras must see the board at the same instant and the overlap zone is
narrow (105-161 simultaneous poses last time).

There is an upper bound too, so "record for ten minutes to be safe" is also wrong: tartancalib on
nine thousand frames is not a long run, it is an abandoned one.

In [ ]:
# ---- on the TX2: nine prompted stages, one bag, with provenance ----
EXPOSURE_US=4986 ./scripts/calib/record_calib_session.sh datasets/calib_2026MMDD
# Preflight refuses to start if the rig would silently produce an unusable recording:
#   trigger_mode must be 1        (else the cameras free-run and sets are not sets)
#   the four channels must share ONE exposure   (the node assumes one exposure per rig)
#   jetson_clocks re-applied, disk checked, systemd-timesyncd stopped for the recording
#     (NTP slewing CLOCK_MONOTONIC made the frame-time fit residual wander 8.4 -> 30.9 us)
# It writes meta.json beside the bag: clock, both stamp conventions, trigger period and
# MEASURED pulse width with its source, decimation, target/layout/noise files, git commit.

# ---- then to the host, and solve offline ----
scp -r tx2-eth:/media/nvidia/workspace/calib_2026MMDD datasets/

# ---- coverage check BEFORE tearing the setup down ----
# This is the one thing the board path cannot show you live. Do it while the rig is still
# set up, so a short stage can be re-recorded rather than re-staged from scratch.
python3 scripts/calib/select_frames.py datasets/calib_2026MMDD/<stage>/cam1 --out /tmp/sel -n 200
#   -> reports cells still short of quota. 13-21 of 64 per camera came back short last time,
#      ALL peripheral, which is exactly where a >180 deg lens is unconstrained.

In [ ]:
# ---- what record_calib_session.sh runs underneath, and the QoS trap ----
# Shown because the trap cost a 204 s recording of nothing, and because the Delta stage in
# section 7 is the same script run again (see 7b: it is run TWICE, at two exposures).
ros2 bag record --qos-profile-overrides-path qos.yaml -o CAM_A \
    /cam1/image_raw /cam1/frame_meta /cam2/frame_meta /cam3/frame_meta /cam4/frame_meta /imu0
# qos.yaml pins the image topics to best_effort. The capture node publishes best-effort on
# purpose (a reliable subscriber can back-pressure the Argus thread); rosbag2 picks its QoS
# from the publishers it can see when it subscribes, so if it subscribes FIRST it asks for
# RELIABLE, the match fails, and it records NOTHING while looking perfectly healthy.

### Verify the image topic, not the file size

A bag growing steadily on IMU data alone passed my first check while recording **zero images**
for 204 seconds. Count the topic that matters:


In [ ]:
import sqlite3, glob
def image_count(bagdir, topic):
    c = sqlite3.connect(glob.glob(bagdir + '/*.db3')[0])
    t = c.execute('SELECT id FROM topics WHERE name=?', (topic,)).fetchone()
    return c.execute('SELECT count(*) FROM messages WHERE topic_id=?', (t[0],)).fetchone()[0] if t else 0

image_count('CAM_A', '/cam1/image_raw')   # must be non-zero within ~10 s of starting


## 2. Decimation must key on the trigger edge, not the frame counter

The capture node publishes 1-in-N to keep the recorder in range. My first version keyed on
each camera's Argus frame *number* - and those counters start when each camera's session
starts, so they are offset between cameras. With N=3, cam1 published edges 0,3,6... while
cam3 published 2,5,8...: **a constant 66.7 ms apart, sharing no instant at all.**

Sensors synchronised to 1 us, and then the software threw away different edges from each.
It cost a whole pairwise recording - 211 seconds with zero simultaneous frames - and was
invisible in the single-camera stages.

The fix keys on the edge index derived from the frame's own SOF time, which every camera
shares to within its 1 us skew:


In [ ]:
// argus_capture_node.cpp
const int64_t period_ns = 1000000000LL / fps_;
const int64_t edge = (int64_t(ft.sof_ns) + period_ns/2) / period_ns;
send = (edge % publish_every_n_) == 0;


## 3. ROS2 -> ROS1

Kalibr is ROS1. `FrameMeta` is excluded rather than registered: Kalibr reads only images and
IMU, and the custom type would just bloat the bag. It stays in the ROS2 originals, which are
the provenance record.


In [ ]:
rosbags-convert --src CAM_A --dst ros1/CAM_A.bag \
                --exclude-msgtype bev_camera/msg/FrameMeta


## 4. Frame filter — and its limitation

Selects a coverage-balanced subset of well-conditioned views. Two reasons: hundreds of
near-identical frames cost hours and add nothing, and a fisheye needs the **periphery**,
which is what a per-cell quota targets (rather than 'seen at least once', which lets one
cell hold a single observation beside another holding a hundred).

**The limitation, learned the hard way:** this counts tags with OpenCV's ArUco. Kalibr's own
AprilGrid detector is far stricter at the periphery, so a frame this passes as '3 tags' can
reach Kalibr's initialiser with 4 corners. Raising the threshold to 8 and then 12 never fixed
the pair solves - see section 6 for what the error actually was.


In [ ]:
# scripts/calib/extract_quarterkalibr_bags.py holds the staged variant; the core selection:
import numpy as np, cv2

GRID, QUOTA = 8, 8

def select(frames, max_frames=220):
    """Greedy against the per-cell DEFICIT: a cell keeps attracting frames until it holds
    QUOTA of them, so coverage comes out even rather than merely non-empty. Ties go to the
    sharper frame, measured on the target's own bounding box - a sharp background at another
    depth says nothing about the tags."""
    need = np.full((GRID, GRID), QUOTA, int)
    picked, pool = [], list(range(len(frames)))
    while pool and len(picked) < max_frames:
        value = lambda i: (sum(min(1, need[r, c]) for (r, c) in frames[i]['cells']),
                           frames[i]['sharpness'])
        best = max(pool, key=value)
        if value(best)[0] == 0:
            break                      # every cell satisfied: stop, do not pad
        for (r, c) in frames[best]['cells']:
            need[r, c] = max(0, need[r, c] - 1)
        picked.append(best); pool.remove(best)
    return picked, need


## 5. Intrinsics

`pinhole-equi` was tried first and **diverged on every camera**, taking 2-3 hours each before
giving up, with the solver's own diagnosis: *"Optimization diverged possibly due to a bad
initialization. (Do the models fit the lenses well?)"* It does not - see the header.

`omni-radtan` (Mei) converges in 1-3 minutes on the same data.


In [ ]:
rosrun kalibr tartan_calibrate \
  --bag /data/ros1f/cam1.bag --topics /cam1/image_raw \
  --models omni-radtan --target /data/april_6x6.yaml \
  --save_dir /data/f_cam1 --dont-show-report


In [ ]:
# results, config/calib/imx296_1456x1088/
#  camera        xi      fx / fy          cx / cy        reproj px
#  cam1 (c FL)  2.172  1695.1 / 1695.3  738.4 / 561.0   0.29 / 0.28
#  cam2 (d FR)  2.111  1647.1 / 1644.7  716.1 / 550.7   0.30 / 0.36
#  cam3 (e BL)  2.156  1687.1 / 1686.8  755.1 / 550.5   0.28 / 0.33
#  cam4 (f BR)  1.908  1546.8 / 1546.1  738.4 / 528.0   0.35 / 0.40
#
# FOV check from the Mei fit: r(theta) = f*sin(theta)/(cos(theta)+xi) peaks at
# cos(theta) = -1/xi -> theta = 117 deg, r_max ~ 875 px, against image corners at ~925 px.
# The equidistant fit agreed independently: fx ~ 536 with detections to r ~ 900 px gives
# theta = 900/536 = 1.68 rad = 96 deg, i.e. ~192 deg full field.


## 6. Extrinsics — use Kalibr's detector, do not reimplement the board

Kalibr's pair solve kept dying in its intrinsics initialiser:

```
RuntimeError: DLT algorithm needs at least 6 points ... 'count' is 5
```

**This is not sparse data.** It is OpenCV's RANSAC drawing 5-point minimal subsets, which its
own DLT then rejects. I misread it as thin detections and spent hours re-filtering recordings
on that premise - tightening thresholds until the left pair fell from 66 usable frames to 29.

The second detour was worse: I reimplemented the board model and omni unprojection by hand.
The camera model round-tripped **exactly** (0.0000 deg over the whole field) and every layout
convention was searched - row/column-major, both flips, all four corner rotations, corner
reversal, and a scan over tag spacing. The residual would not go below 11 px, and the
extrinsic came out with a **1 m baseline on a 15 cm rig**.

The fix was to stop guessing at a correspondence Kalibr already defines
(`GridCalibrationTargetAprilgrid.cpp`) and call its API: same target object, same detector,
same camera geometry, and its own `T_t_c` per view.


In [ ]:
import numpy as np, rosbag, aslam_cv as acv, aslam_cameras_april as acv_april
import kalibr_common as kc
from cv_bridge import CvBridge

def detector_for(chain_yaml, grid):
    chain = kc.ConfigReader.CameraChainParameters(chain_yaml)
    cam = kc.AslamCamera.fromParameters(chain.getCameraParameters(0))
    o = acv.GridDetectorOptions(); o.filterCornerOutliers = False
    return acv.GridDetector(cam.geometry, grid, o)

tp = kc.ConfigReader.CalibrationTargetParameters('april_6x6.yaml').getTargetParams()
opts = acv_april.AprilgridOptions()
opts.minTagsForValidObs = int(max(tp['tagRows'], tp['tagCols']) + 1)   # = 7, as Kalibr does
grid = acv_april.GridCalibrationTargetAprilgrid(tp['tagRows'], tp['tagCols'],
                                                tp['tagSize'], tp['tagSpacing'], opts)

def poses(bag, topic, det):
    """stamp -> T_target_camera, keyed on the frame's OWN header stamp (never arrival time:
    matching on bag timestamps reported ZERO simultaneous pairs on a rig triggered to 1 us)."""
    out, bridge = {}, CvBridge()
    for _, msg, _ in rosbag.Bag(bag).read_messages(topics=[topic]):
        img = bridge.imgmsg_to_cv2(msg, desired_encoding='mono8')
        ok, obs = det.findTarget(acv.Time(msg.header.stamp.secs, msg.header.stamp.nsecs),
                                 np.array(img))
        if ok:
            out[msg.header.stamp.to_nsec()] = obs.T_t_c().T()
    return out

# extrinsic = inv(T_target_camB) @ T_target_camA, averaged over simultaneous views
# (rotation averaged by SVD projection of the mean matrix)


In [ ]:
# results, config/rig/rig_extrinsics_imx296.yaml
#  pair                 poses  baseline   rotation  spread
#  left  cam3 -> cam1     153   147.7 mm    90.9 deg  0.55 deg
#  front cam1 -> cam2     161   148.7 mm    90.4 deg  0.68 deg
#  right cam2 -> cam4     105   149.2 mm    92.2 deg  0.65 deg
#  rear  cam4 -> cam3     150   149.2 mm    90.1 deg  0.55 deg
#
# Four INDEPENDENT solves, from separate recordings, agreeing on baseline to 1.5 mm.
# Ring closure cam3->cam1->cam2->cam4->cam3:  4.75 deg rotation, 4.9 mm translation
# over a ~0.6 m loop (0.8%). Rotation is the weaker half: the hops sum to 363.6 deg and
# the right pair, with the fewest poses, is the outlier at 92.2.


## 6b. Ring closure — four free edges become one rigid body

The four pairs are solved independently, so nothing makes them describe a single object. Walking
`cam3 -> cam1 -> cam2 -> cam4 -> cam3` should return to the identity and does not: **3.63 deg
rotation, 9.2 mm translation**. `close_rig_ring.py` re-parameterises to three camera poses in
cam1's frame (18 dof instead of 24) and runs Levenberg-Marquardt, taking the residual to
**0.0000 deg / 0.043 mm** with per-edge corrections of 0.57-1.31 deg.

**It buys consistency, not accuracy, and that was established before running it.** A Monte-Carlo
over each pair's own reported spread says random error would leave only ~0.5 deg of loop residual,
so the 3.63 deg is systematic bias inside each recording and averaging is already saturated.
Measured cost, per-pair epipolar median before -> after: left 0.81 -> 0.45, front 1.42 -> 1.79,
right 0.62 -> 2.55, rear 0.83 -> 0.66; mean rms 2.94 -> 3.06 px. Rigidity costs ~4% in rms and
**moves error between pairs** - worth it against 3.63 deg (35 px at the image edge), but a trade.

Two guards, both learned here:

- **A closed loop is not evidence of a correct answer.** Plain Gauss-Newton drove the residual to
  zero while walking the rig to 70-115 deg per edge. That is why the script prints per-edge
  corrections: a correction far outside each pair's own angular spread is the solver hiding a
  bad edge in a good-looking loop.
- Weighting by measured epipolar residual instead of internal angular spread was tried and is
  **worse** (mean rms 3.45). Kept as an option, not the default.

In [ ]:
python3 scripts/calib/close_rig_ring.py \
  --extrinsics <session>/rig_ext.yaml --out <session>/closed.yaml
# check, in this order:
#   1. residual after  ~ 0.000 deg / < 0.1 mm
#   2. per-edge corrections vs each pair's own angular spread (0.55-0.68 deg last time)
#   3. the angle between facing virtual optical axes: 1.1/1.3/2.1/1.3 -> 1.1/1.4/1.1/1.0 deg.
#      Nothing in the closure targeted that number, so it is independent confirmation - the
#      right pair's outlier turned out to be the inconsistency, not the geometry.

## 6c. Virtual stereo — the carve cuVSLAM actually consumes

cuVSLAM's only fisheye model is equidistant, capped **below 180 deg**, and these lenses fit
~192 deg (see the header). So each fisheye is carved into **two virtual pinholes at +/-45 deg**
and the VO is fed **eight Pinhole cameras**, not four fisheyes. The facing pair of each stereo
pair is *derived* from the extrinsic rather than assumed.

**768x576, fov 70 deg, focal 548.4 px** - and the fov comes from the lens's **horizontal 160**,
not its diagonal 190. The first attempt carved by the diagonal, which asks each pinhole to reach
95 deg off-axis where the lens delivers 80: every rectified view came back with a black wedge
(90% non-black, against 100% now). The lens is D190/**H160** and the split is by yaw, so the
horizontal field governs.

Two measurement traps: **resolution is not free** - at 480x360 the virtual focal is 0.38x the
fisheye's near-axis scale and tag detection collapsed from ~10/frame to 0.1. And **ORB-based
epipolar measurement is worthless here** - 4.15 px median / 74 px p90 on a repetitive tag scene,
measuring the matcher rather than the rig. Tag identity removes the ambiguity.

Dense disparity on a calibration sweep is **not evidence either way**: a repetitive grid at
0.3-0.6 m against blank wall and floor is close to the worst case for block matching, and
widening the search made it worse, so texture is the limit rather than geometry. A real verdict
needs a textured scene at 1-3 m.

In [ ]:
# regenerate all four pairs on the closed rig and report epipolar residual, measured vs closed
docker run -v <session>:/data -v $PWD:/repo tartancalib:latest bash /repo/scripts/calib/regen_vstereo.sh
# gates: 100% non-black per rectified view | disparity sign consistent | median |dy| per pair
#   last time: left 0.45 | front 1.79 | right 2.55 | rear 0.66 px
# Usable, not excellent - a good physical stereo rig reaches under 0.5 px, so the disparity
# search needs a few pixels of vertical tolerance.

# then: does cuVSLAM actually pair these cameras?  It does not take declared stereo pairs - it
# samples a grid per camera, back-projects to 2 m and 4 m, and connects pairs exceeding a
# HARD-CODED 0.5 (frustum_intersection_graph.cpp:33).  Re-run its own test on our poses:
./scripts/vo/verify_rig_build.sh      # 0.939 / 0.951 / 0.926 / 0.949 against a 0.961 ceiling
# Run this BEFORE the board session: a sign error in rig_from_fisheye * Ry still yields a rig
# cuVSLAM accepts while finding no stereo pairs at all (pairing drops to ~0.03), and on the
# board that reads as a wiring bug rather than a geometry bug.

## 7. Camera-IMU offset (Delta)

Board **fixed**, rig **moved** - Kalibr recovers Delta by comparing the motion the camera infers
from a static target against the motion the IMU measured.

**The excitation is the measurement.** Rotate about all three axes, then translate along all
three, brisk (~1-2 Hz) but not hard enough to smear tags, and keep the target in view *while*
the IMU is excited. Delta is observable only where both hold; a gentle wave returns a confident
wrong number.

**Delta is measured AT an exposure.** The stamp is `SOF - exposure/2`, so a Delta fitted at one
pulse width does not transfer to another: calibrate at 30 ms and fly at 5 ms and every camera
stamp is **12.5 ms** out, against a Delta of -8.06 ms and a 1 us sync budget. Record this stage
at the exposure the VO will actually run at.

Two problems had to be worked around:

1. **The bag held only 43 Hz of IMU.** rosbag2's single writer thread cannot carry full-rate
   images and 200 Hz IMU together, so the surplus was dropped from the publisher queue. The IMU
   node writes every sample to CSV regardless, at the data-ready edge on the same clock, so the
   input was rebuilt from that: 19717 samples at **199.5 Hz** over the 96.8 s image span. (An
   earlier "204 Hz" was IMU count over the *image* window - arithmetic, not a rate.) **Check
   `imu0.csv` before tearing the rig down.**
2. **Kalibr's own version bug** - `aopt.NoMEstimator()` called with no argument against a backend
   requiring a double. Patched in place; it is a no-op estimator, so the value is irrelevant.

In [ ]:
sed -i 's/NoMEstimator()/NoMEstimator(1.0)/g' $(grep -rl 'NoMEstimator()' /catkin_ws)
rosrun kalibr kalibr_calibrate_imu_camera \
  --bag /data/ros1/CAM_IMU_full.bag --cam /data/intr_cam1.yaml \
  --imu /data/imu.yaml --target /data/april_6x6.yaml --dont-show-report


In [ ]:
# MEASURED 2026-09-01 (cam1, 4986 us exposure, 1101 images, 14693 IMU samples at 200.11 Hz)
# timeshift_cam_imu: +0.003731525 s   (Kalibr's convention: t_imu = t_cam + shift)
# residuals: 0.45 px reprojection
#
# The 2026-08-28 value of -8.06 ms DOES NOT REPRODUCE, and should not be used. That solve
# ran on an IMU stream rebuilt from CSV (rosbag2 had dropped it to 43 Hz) and on intrinsics
# since shown to be unconverged. Recording on the HOST fixed the IMU rate; the corner
# re-sweeps fixed the intrinsics.
#
# REPEATABILITY - the solver reports none, so measure it by repeating:
#   +3.7315 ms   /  +3.7556 ms   two independent recordings, same settings  -> 24 us apart
#
# WHAT IT IS MADE OF - all three terms measured, none assumed:
#   +2.90 ms   gyro DLPF group delay (184 Hz), reported by the driver and NOT applied.
#              Proven by varying it: 184 -> 41 Hz adds 3.00 ms of datasheet delay and moved
#              Delta by +2.79 ms. That slope is what identifies the mechanism.
#   -0.26 ms   camera side, from the hardware trigger echo (section 7c)
#   ~+1.1 ms   unattributed; the accel path (1.88 ms, a different filter) is the suspect
#
# ⚠ accel_dlpf=2 (99 Hz, 2.88 ms) would align the accel to the gyro's 2.90 ms - 0.02 ms
#   instead of the present 1.02 ms mismatch, which one Delta cannot correct. Re-measure
#   Delta if you change it. See config/calib/imu_mpu9250.yaml.

### 7c. The trigger echo — measuring the camera half directly

Everything above estimates Delta with a solver, and `WIRING.md` is blunt about the weakness:
Route B "cannot be checked against anything". Route A can. One trigger channel is echoed into a
Tegra GPIO and timestamped, so the camera's claim is compared against the **real edge**.

**The wiring needs no components**, which is what makes it worth doing. The Tegra pad
(`GPIO_PQ5_PI5` = `gpio-389`, M110 `J21` pin 8) is **1.8 V unbuffered** and the STM32 drives 3.3 V,
so a direct tap would destroy it and a divider needs resistors. Instead the F401 drives a *second*
copy of `TIM2_CH1` on **`PA5` as OPEN-DRAIN**: it can only pull the line down, and the high level
comes from a Tegra internal pull-up enabled in the DTB. Nothing on that wire ever exceeds 1.8 V.

  * firmware: `hw-trigger/firmware-f401/Core/Src/stm32f4xx_hal_msp.c` (PA5, `GPIO_MODE_AF_OD`)
  * device tree: `gpio_pq5_pi5` with `nvidia,pull = <2>`, booted as `LABEL j106echo`
  * verification before wiring: `gpio-389` read a stable **0** on the old DTB and **1** on the new

Measure the FALLING edge - the one the open-drain pin drives hard, and under `pol 0` also the
start of exposure. The rising edge is a weak pull-up charging a wire.

In [ ]:
# on the board
sudo ./trig-echo-stamp.py --seconds 30 --out /tmp/echo.csv      # timestamps falling edges
./trig-echo-delta.py --echo /tmp/echo.csv --frames <data>/cam1.csv

#   e = t_sof - t_edge - exposure        0 if SOF marks the end of exposure
#     5 ms exposure ->  +0.3112 ms
#    10 ms exposure ->  +0.3097 ms
#    20 ms exposure ->  +0.3109 ms
#
# A CONSTANT +0.31 ms, varying 1.5 us over a 4x exposure change. So SOF sits a fixed
# 0.31 ms after end-of-exposure, the exposure/2 stamping is right, and the camera
# contributes only w - e = -0.26 ms to Delta. Everything else is on the IMU side.
#
# Transport was excluded separately: camera messages arrive ~55 ms later relative to their
# stamps than IMU samples (1.58 MB vs 50 bytes), but BOTH are stamped at source, so DDS
# latency cannot reach Delta - and if it could the signature would be 55 ms, not 4.

### 7a. Verify Delta is an offset, not a fitting artefact — inject a known shift

Kalibr will return *a* number from almost any recording. The check that it is the **timestamp
offset** and not the solver absorbing something else: shift every camera stamp by a known amount,
re-solve, and confirm the estimate moves by exactly minus that amount.

`+10.000 ms` moved the estimate by `-10.000 ms`, residual **-1.2 us**. That also disposes of the
quantisation worry: Kalibr fits a continuous B-spline, so a 200 Hz IMU does not quantise Delta -
8 ms is 1.6 samples and nothing about the rate forbids it.

**Do this every time.** It costs one extra solver run against a bag you already have.

In [ ]:
python3 <session>/filter_bag.py --in ros1/CAM_IMU_full.bag --out ros1/CAM_IMU_shift10.bag \
        --shift-camera-stamps +0.010
rosrun kalibr kalibr_calibrate_imu_camera --bag /data/ros1/CAM_IMU_shift10.bag \
  --cam /data/intr_cam1.yaml --imu /data/imu.yaml --target /data/april_6x6.yaml --dont-show-report
# expect:  timeshift(shifted) - timeshift(full)  ==  -0.010000 s   (we got -1.2 us of residual)

### 7b. Two exposures — what actually causes the -8.06 ms

The offset is larger than the stamping discipline should leave, and there are exactly two
candidate mechanisms. They matter differently: **a readout offset is CONSTANT and Delta absorbs
it permanently, while a mis-modelled exposure MOVES with the trigger pulse width** - and the
pulse width *is* the exposure on this rig.

This was blocked on wiring the trigger echo into a GPIO (`hw-trigger/WIRING.md` 4.4). It no
longer is: the two hypotheses make **opposite predictions** about a quantity the F401 varies at
will.

| record the camera-IMU stage twice, identical but for exposure | then |
|---|---|
| Delta **invariant** | the exposure term is modelled correctly, and -8.06 ms is the constant readout offset - half the J106-measured 16.1 ms readout is 8.05 ms, a 10 us coincidence |
| Delta **moves** | mis-modelled; a half-width error predicts `dDelta = (E1 - E2)/2`, and the slope says how much |

Use **5 ms vs 15 ms**, not 5 vs 30: the prediction is a **5 ms** shift, still ~4000x Kalibr's
demonstrated resolution on this data, and it avoids blurring the tags during exactly the vigorous
motion this stage requires. (30 ms also clips cam2 and cam4 on the windows in this room.)

Fold the result back into task 3.6b and, if it lands, close it - the GPIO echo becomes
confirmation rather than the only route.

In [ ]:
# Record the SAME stage twice, changing only the exposure. Command the generator, then READ
# BACK the pulse width and stamp with that - the commanded and asserted values differ by ~14 us
# (5000 -> 4985740 ns), and the whole point of section 0 is not to assume which one is in force.
for CMD in 5000 15000; do
  ssh tx2-eth "python3 /home/nvidia/tools/j106-trigctl.py --port /dev/ttyACM0 exposure $CMD"
  PULSE=$(ssh tx2-eth "python3 /home/nvidia/tools/j106-trigctl.py --port /dev/ttyACM0 status" \
          | grep -oP 'ch1_exposure_us=\d+ pulse_ns=\K\d+')
  E=$(( (PULSE + 500) / 1000 ))
  EXPOSURE_US=$E ./scripts/calib/record_calib_session.sh datasets/calib_2026MMDD/camimu_$E
done
# solve both, subtract.  Expected if the exposure term is mis-modelled: dDelta = (E1-E2)/2 = 5 ms.

## 8. What to do differently next time

| | |
|---|---|
| **Select offline, not live** | live decimation keeps every Nth frame whether or not it is any good. The ROS path caps at ~9-11 Hz on rosbag2's single writer thread, so decimate on the TRIGGER EDGE (section 2) and let `select_frames.py` choose. An MJPEG stream to the host lifts that cap but carries no capture timestamps, so it cannot serve the pair stages - section 1 |
| **Sweep the corners deliberately** | 13-21 of 64 cells per camera stayed short of quota, all peripheral - which is exactly where a >180 deg lens needs constraint |
| **Add a stage where 3+ cameras see the board** | four independent pairwise recordings cannot see the ~1 deg of systematic per-recording bias that dominates the ring residual; the closure absorbs it silently. **Now stage 9** |
| **Calipers on the printed target** | a print-scale error is common-mode across every recording, invisible to ring closure and to every consistency check run so far, and it scales VO translation *directly*. Never checked on the 2026-08-28 board |
| **Verify the image topic, not the byte count** | one bag recorded 204 s of zero images while growing steadily on IMU data |
| **Restart capture after any `nvargus-daemon` restart** | `/tmp/argus_socket` is a *file*; restarting the daemon replaces it and orphans any container that bind-mounted it. The node stays alive and silently stops delivering |
| **Read the library's source before reimplementing its conventions** | the board correspondence cost hours of guessing; `GridCalibrationTargetAprilgrid.cpp` answers it in 15 lines |
| **Run the solver container as `--user $(id -u):$(id -g)`** | the 2026-08-28 outputs came back root-owned, so cleaning up 5.6 GB of solver state needed `sudo` on a path that should never have needed it |
| **Measure the trigger, never read its label** | polarity flipped `active_high` -> `active_low` across the generator swap with no change in meaning, and Argus's exposure was wrong by ~10x. Both are settled by a two-minute sweep, and both silently bias every timestamp otherwise |